Proyecto para deteccion de objetos por : 
-Reciclable
-No Reciclable

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras import optimizers # Optimizadores para el gradiente descendiente
from keras.models import Sequential #Definir la arquitectura de la red neuronal multicapa secuencial
from keras.layers import Dense,Dropout,Flatten,Activation 
#Dense=define cada capa, 
# Dropout= evita el sobreajuste, 
# Flatten= aplana los datos de una matriz a un vector, 
# Activation= funciones de activación
from keras.layers import Convolution2D,MaxPooling2D #Datos en forma de matriz

In [ ]:
# Rutas de imágenes
entrenamiento = 'CNN/Imagenes/TRAIN'
validacion = 'CNN/Imagenes/VALIDATION'

# Hiperparámetros
epocas = 30
altura, anchura = 200, 200
batch_size = 2
pasos = 100

# Parámetros de capas
kernel1 = 32
kernel1_size = (3, 3)
kernel2 = 64
kernel2_size = (3, 3)
size_pooling = (3, 3)
clases = 2  # N (no reciclable), R (reciclable)

In [ ]:
# Generadores
entrenar = ImageDataGenerator(
    rescale=1/255,
    zoom_range=0.20,
    horizontal_flip=True
)

validar = ImageDataGenerator(rescale=1/255)

# Leer imágenes
imagenes_entrenamiento = entrenar.flow_from_directory(
    entrenamiento,
    target_size=(altura, anchura),
    batch_size=batch_size,
    class_mode='categorical'
)

imagenes_validacion = validar.flow_from_directory(
    validacion,
    target_size=(altura, anchura),
    batch_size=batch_size,
    class_mode='categorical'
)

In [ ]:
#definir la arquitectura de la red neuronal convolucional

CNN=Sequential()

#Definir la primera capa convolucional- copiar y pegar esto 10 veces si tengo 10 capas
CNN.add(Convolution2D(kernel1,kernel_size=kernel1_size,
                      padding='same',
                      input_shape=(altura,anchura,3),
                      activation='relu'))
CNN.add(MaxPooling2D(pool_size=size_pooling))

#Definir la segunda capa convolucional
CNN.add(Convolution2D(kernel2,kernel_size=kernel2_size,
                      padding='same',
                      input_shape=(altura,anchura,3),
                      activation='relu'))
CNN.add(MaxPooling2D(pool_size=size_pooling))

#se conecta el proceso de convolucion con el perceptrón multicapa
CNN.add(Flatten()) #aplanar las matrices para formar el vector de caracteristicas

#definir aqui el perceptrón multicapa
#primera capa oculta
CNN.add(Dense(255,activation='relu'))
#segunda capa oculta
CNN.add(Dense(255,activation='relu'))

#EVitar sobrentrenamiento con DropOut (Apagar un porcentaje de las neuronas)
CNN.add(Dropout(0.5))

#capa de salida
CNN.add(Dense(2,activation='softmax'))


# Arquitectura CNN
CNN = Sequential()

# Primera capa convolucional
CNN.add(Convolution2D(kernel1, kernel_size=kernel1_size,
                      padding='same',
                      input_shape=(altura, anchura, 3),
                      activation='relu'))
CNN.add(MaxPooling2D(pool_size=size_pooling))

# Segunda capa convolucional
CNN.add(Convolution2D(kernel2, kernel_size=kernel2_size,
                      padding='same',
                      activation='relu'))
CNN.add(MaxPooling2D(pool_size=size_pooling))

# Aplanar y conectar con el perceptrón
CNN.add(Flatten())

# Capas ocultas
CNN.add(Dense(255, activation='relu'))
CNN.add(Dense(255, activation='relu'))

# Dropout para evitar sobreajuste
CNN.add(Dropout(0.5))

# Capa de salida
CNN.add(Dense(clases, activation='softmax'))

In [ ]:
CNN.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["acc", "mse"])

CNN.fit(imagenes_entrenamiento,
        validation_data=imagenes_validacion,
        epochs=epocas,
        validation_steps=pasos,
        verbose=1)

In [ ]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
from keras.models import load_model
import os

# Guardar modelo y pesos
CNN.save("CNN/Imagenes/Modelo/cnn.h5")
CNN.save_weights("CNN/Imagenes/Modelo/pesos.weights.h5")

# Imagen a clasificar (puedes cambiarla por cualquiera de TEST)
imagen_clasificar = 'CNN/Imagenes/TEST/R/tu_imagen.jpg'

# Cargar modelo
cnn = load_model("CNN/Imagenes/Modelo/cnn.h5")
cnn.load_weights("CNN/Imagenes/Modelo/pesos.weights.h5")

# Preparar imagen
evaluar_imagen = load_img(imagen_clasificar, target_size=(altura, anchura))
evaluar_imagen = img_to_array(evaluar_imagen)
evaluar_imagen = np.expand_dims(evaluar_imagen, axis=0)

# Predecir
clase = cnn.predict(evaluar_imagen)
print(clase)

arg_max = np.argmax(clase[0])

if arg_max == 0:
    print("No reciclable ♻️✗")
elif arg_max == 1:
    print("Reciclable ♻️✓")